In [1]:
import os
import dspy
import pandas as pd
from dotenv import load_dotenv
from groq import Groq
from pydantic import BaseModel

load_dotenv()

/home/ash/miniconda3/envs/nanites/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
client = Groq(api_key = os.getenv("GROQ_API_KEY"))
lm_gpt4o = dspy.LM("openai/gpt-4o", temperature=0.9, api_key=os.getenv("OPENAI_API_KEY"))
lm_llama70b_r1 = dspy.LM(
                "openai/deepseek-r1-distill-llama-70b",
                api_key=os.getenv("GROQ_API_KEY"),
                api_base="https://api.groq.com/openai/v1",
            )
dspy.configure(lm=lm_gpt4o)

In [3]:
pcap_path = "/home/ash/github/packet_analysis/csv/radius_localhost_vikram.csv"
df = pd.read_csv(pcap_path)
columns = df.columns.tolist()[1:]
print(columns)
df_head = df.head(3).to_markdown()
print(df_head)

['eap.identity', 'ip.dsfield.dscp', 'udp.length', 'udp.checksum', 'eap.id', 'udp.time_relative', 'udp.srcport', 'frame.time_epoch', 'frame.time_delta', 'eap.type', 'ip.dsfield', 'radius.authenticator', 'radius.eap_fragment', 'frame.interface_name', 'ip.dsfield.ecn', 'udp.dstport', 'null.family', 'frame.len', 'ip.flags.rb', 'ip.addr', 'frame.section_number', 'ip.src_host', 'udp.port', 'ip.src', 'ip.version', 'ip.id', 'radius.code', 'eap.len', 'ip.ttl', 'frame.marked', 'frame.encap_type', 'udp.payload', 'frame.time_utc', 'ip.hdr_len', 'frame.ignored', 'udp.checksum.status', 'radius.avp', 'ip.flags', 'ip.dst', 'eap.code', '_index', 'ip.len', '_type', 'ip.checksum', 'ip.flags.df', 'ip.proto', 'frame.protocols', 'ip.dst_host', 'frame.time', 'frame.interface_id', 'ip.flags.mf', 'udp.stream', 'udp.time_delta', 'frame.time_relative', 'ip.frag_offset', 'radius.length', '_score', 'ip.host', 'ip.checksum.status', 'radius.req', 'frame.cap_len', 'frame.offset_shift', 'radius.id', 'frame.time_delta_

#### PCAP to CSV

In [4]:
# Define a simple signature for basic question answering
class BasicQA(dspy.Signature):
    """Answer questions with short factoid answers."""
    context = dspy.InputField(desc="a paragraph of text")
    question = dspy.InputField()
    answer = dspy.OutputField(desc="column names are: ...")

# Pass signature to ReAct module
react_module = dspy.ReAct(BasicQA, tools=[])
# COT
cot = dspy.ChainOfThought(BasicQA)

In [5]:
# Call the ReAct module on a particular input
query = "What are the user names in the pcap file?"

def get_column_names(query, df_head):
    question_parser =  dspy.ChainOfThought("question -> answer")

precontext = f"return all possible column names, the more the better that would be needed to answer the question"
context = f""" You are provided a pcap file in csv format. These are the columns in the file with first 3 rows:
                {df_head}
            """


question = f"{precontext}: {query}"
result_react = react_module(context=context, question=question)
result_cot = cot(context=context, question=question)

print(f"Question: {question}")
print(f"Final Predicted Answer (after ReAct process): {result_react.answer}")
print(f"Final Predicted Answer (after COT process): {result_cot.answer}")

Question: return all possible column names, the more the better that would be needed to answer the question: What are the user names in the pcap file?
Final Predicted Answer (after ReAct process): eap.identity, radius.User_Name
Final Predicted Answer (after COT process): Column names are: eap.identity, radius.User_Name, radius.eap_fragment


## Answer:
```
Question: Based on these column names, retrieve only the columns that might contain information regarding the question: 'Are there any viruses in the dataset?'
Final Predicted Answer (after ReAct process): ['eap.mitm_attacks', '_ws.expert.group', '_ws.expert.message', '_ws.expert.severity']
```


In [7]:
lm_gpt4o.inspect_history()





[2025-01-29T13:47:19.562770]

System message:

Your input fields are:
1. `context` (str): a paragraph of text
2. `question` (str)

Your output fields are:
1. `reasoning` (str)
2. `answer` (str): column names are: ...

All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## context ## ]]
{context}

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]

In adhering to this structure, your objective is: 
        Answer questions with short factoid answers.


User message:

[[ ## context ## ]]
 You are provided a pcap file in csv format. These are the columns in the file with first 3 rows:
                |    |   Unnamed: 0 | eap.identity   |   ip.dsfield.dscp |   udp.length | udp.checksum   |   eap.id |   udp.time_relative |   udp.srcport |   frame.time_epoch |   frame.time_delta |   eap.type | ip.dsfield   | radius.authenticator                            | radius.eap_fra

## Algorithm starts here:

In [1]:
import os
import dspy
import pandas as pd
import re
import ast
import json
from dotenv import load_dotenv
from groq import Groq
from pydantic import BaseModel
from openai import OpenAI
from pydantic import BaseModel

load_dotenv()

groq = Groq(api_key = os.getenv("GROQ_API_KEY"))
gpt = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
pcap_path = "/home/ash/github/packet_analysis/csv/radius_localhost_vikram.csv"
df = pd.read_csv(pcap_path)
columns = df.columns.tolist()[1:]
print(columns)
df_head = df.head(3).to_markdown()
print(df_head)

/home/ash/miniconda3/envs/nanites/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['eap.identity', 'ip.dsfield.dscp', 'udp.length', 'udp.checksum', 'eap.id', 'udp.time_relative', 'udp.srcport', 'frame.time_epoch', 'frame.time_delta', 'eap.type', 'ip.dsfield', 'radius.authenticator', 'radius.eap_fragment', 'frame.interface_name', 'ip.dsfield.ecn', 'udp.dstport', 'null.family', 'frame.len', 'ip.flags.rb', 'ip.addr', 'frame.section_number', 'ip.src_host', 'udp.port', 'ip.src', 'ip.version', 'ip.id', 'radius.code', 'eap.len', 'ip.ttl', 'frame.marked', 'frame.encap_type', 'udp.payload', 'frame.time_utc', 'ip.hdr_len', 'frame.ignored', 'udp.checksum.status', 'radius.avp', 'ip.flags', 'ip.dst', 'eap.code', '_index', 'ip.len', '_type', 'ip.checksum', 'ip.flags.df', 'ip.proto', 'frame.protocols', 'ip.dst_host', 'frame.time', 'frame.interface_id', 'ip.flags.mf', 'udp.stream', 'udp.time_delta', 'frame.time_relative', 'ip.frag_offset', 'radius.length', '_score', 'ip.host', 'ip.checksum.status', 'radius.req', 'frame.cap_len', 'frame.offset_shift', 'radius.id', 'frame.time_delta_

In [2]:
lm_gpt4o = dspy.LM("openai/gpt-4o", temperature=0.9, api_key=os.getenv("OPENAI_API_KEY"))
lm_llama70b_r1 = dspy.LM(
                "openai/deepseek-r1-distill-llama-70b",
                api_key=os.getenv("GROQ_API_KEY"),
                api_base="https://api.groq.com/openai/v1",
            )
dspy.configure(lm=lm_gpt4o)
class ColumnSelectorQA(dspy.Signature):
    """Answer questions with short factoid answers."""
    context = dspy.InputField(desc="a paragraph of text")
    question = dspy.InputField()
    answer = dspy.OutputField(desc="list of column names")

class RouterQA(dspy.Signature):
    """Answer questions with short factoid answers."""
    question = dspy.InputField(desc=" task description")
    answer = dspy.OutputField(desc="`more_context` or `dataframe_operation`")

class CodingQA(dspy.Signature):
    """Answer questions with short factoid answers."""
    question = dspy.InputField()
    answer = dspy.OutputField(desc="code snippet list")

class DescriptionQA(dspy.Signature):
    """Answer in descriptive format"""
    context = dspy.InputField(desc="a paragraph of text with pcap in csv format")
    question = dspy.InputField()
    answer = dspy.OutputField(desc="concise and non generic answer rendered with a heading and in markdown format")

class GotAnswerJudge(dspy.Signature):
    """Judge if the answer is factually correct based on the context."""
    question = dspy.InputField(desc="Question to be answered")
    answer = dspy.InputField(desc="Answer for the question")
    factually_correct = dspy.OutputField(desc="Is the question addressed completely by the answer?", prefix="Factual[Yes/No]:")

class LongOrShortJudge(dspy.Signature):
    """Judge if the answer is to be descriptive or can be done in one step."""
    question = dspy.InputField(desc="Question to be answered")
    long_or_short = dspy.OutputField(desc="Will a precise answer satisfy the user(i.e do they have a specific goal in the question) or a long descriptive one (i.e are they exploring)?", prefix="Long or Short")

class Task:
    def __init__(self, df, user_query):
        self.df = df
        self.user_query = user_query
        self.columns = df.columns.tolist()[1:]
        self.column_cot = dspy.ChainOfThought(ColumnSelectorQA)
        self.router_cot = dspy.ChainOfThought(RouterQA)
        self.coding_cot = dspy.ChainOfThought(CodingQA)
        self.description_cot = dspy.ChainOfThought(DescriptionQA)
        self.long_or_short_judge = dspy.ChainOfThought(LongOrShortJudge)
        self.steps_eval = []
        self.type = None

    def _get_column_names(self, query):
        try:
            context = f""" You are provided a pcap file in csv format. These are the column names in the csv file:
                            str({self.columns})
                        """
            question = f"Return all possible column names with at most 10 columns to fulfil this task: {query}"
            result_cot = self.column_cot(context=context, question=question)
            try:
                columns = ast.literal_eval(result_cot.answer)
                if isinstance(columns, list):
                    return columns
            except Exception as e:
                print("Could not parse the column names to list-> main")
                print("error", e)
                return []
        except Exception as e:
            print(f"Error in _get_column_names: {str(e)}")
            return []

    def _filter_columns(self, columns, query):
        try:
            context = f""" You are provided a pcap file in csv. These are the column names in the csv file:
                            str({columns})
                        """
            question = f"Return the most important columns from the context that would be required to extract information for this task: {query}"
            result_cot = self.column_cot(context=context, question=question)
            try:
                columns_filtered = ast.literal_eval(result_cot.answer)
                if isinstance(columns_filtered, list):
                    return columns_filtered
            except Exception as e:
                print("Could not parse the column names to list-> filtered")
                return columns[:10]  # Return first 10 columns as fallback
        except Exception as e:
            print(f"Error in _filter_columns: {str(e)}")
            return columns[:10]

    def more_context(self, query):
        limit = 5
        try:
            columns = self._get_column_names(query)
            if not columns:  # If no columns returned, use first 10 columns
                columns = self.columns[:limit]
            if len(columns) > limit:
                columns = self._filter_columns(columns, query)
            
            # Ensure we have valid columns before creating markdown
            valid_columns = [col for col in columns if col in self.df.columns]
            if not valid_columns:
                valid_columns = self.df.columns[:limit]
            print("valid_columns", valid_columns)
            df_in_markdown = self.df.loc[:, valid_columns].to_markdown()
            print("df_in_markdown", df_in_markdown)
            context = f""" You are provided a pcap file with only some of the columns shown below:
                            {df_in_markdown}
                        """
            if self.type == "short":
                question = f"""[Question]:{self.user_query} 
                            [IMPORTANT]: DO NOT MENTION 'CSV' or 'PANDAS' in the answer, only refer to the data as a pcap."""
            elif self.type == "long":
                question = f"""You are tasked on expounding on the following task:{query}, with the main goal to answer the question:{self.user_query} 
                            Based on the new information provided in context, return a detailed answer regarding the technical aspects of the task.
                            [IMPORTANT]: DO NOT MENTION 'CSV' or 'PANDAS' in the answer, only refer to the data as a pcap."""
                
            description = self.description_cot(context=context, question=question)
            return description.answer
        except Exception as e:
            print(f"Error in more_context: {str(e)}")
            return f"Unable to process the query due to an error: {str(e)}"


    def router(self, task: str):
        try:
            method_name = "more_context"
            if hasattr(self, method_name):
                method = getattr(self, method_name)
                try:
                    description = method(task)
                    result = {"task": task, 
                              "answer": description,
                              "error": False
                        }
                    self.steps_eval.append(result)
                except Exception as e:
                    error_result = {
                        "task": task,
                        "answer": f"Error processing task: {str(e)}",
                        "error": True
                    }
                    self.steps_eval.append(error_result)
                    print(f"Error executing {method_name}: {str(e)}")
            else:
                error_result = {
                    "task": task,
                    "answer": f"Method {method_name} does not exist",
                    "error": True
                }
                self.steps_eval.append(error_result)
                print(f"Method {method_name} does not exist")
        except Exception as e:
            error_result = {
                "task": task,
                "answer": f"Fatal error in router: {str(e)}",
                "error": True
            }
            self.steps_eval.append(error_result)
            print(f"Fatal error in router: {str(e)}")

    def execute(self, steps):
        question = self.user_query
        context = "\n\n".join(steps)
        long_or_short = self.long_or_short_judge(question=question, context=context)
        print("user query goal", long_or_short.long_or_short)
        if "long" not in long_or_short.long_or_short.lower():
            context = ["\n\n".join(steps)]
            self.type = "short"
            return context
        else:
            self.type = "long"
            return steps

### Master node:

In [3]:
def step_outliner(user_input):
    total_packets = len(df)
    df_head = df.head(3).to_markdown()
    context = f""" You are provided a pcap file in csv format. There are total of {total_packets} packets (rows of the csv), but only first 3 rows are shown below:
                    {df_head}
                    from the context answer the following question :
                """
    question = f"""
                [user profile]: A network engineer is asking you a question about the pcap file(s).
                [question]: {user_input}
                [Note]: provide a list of steps, (highlighted by `###`), to get the answer. Do not write any code. Just provide the required column names, whose entire context would be needed to answer the question.
                """
    query = f"{context}: \n\n{question}"

    completion = groq.chat.completions.create(
        model="deepseek-r1-distill-llama-70b",
        messages=[
            {
                "role": "user",
                "content": query,
            },
        ],
        temperature=0.6,
        top_p=0.95,
        stream=True,
        stop=None,
    )

    full_response = ""
    for chunk in completion:
        content = chunk.choices[0].delta.content or ""
        full_response += content  # Append each chunk to the full response
    return (full_response)  # Optional: still print while storing

def is_serialized(text):
    # Check if a line contains a number followed by a period"
    return bool(re.match(r'.*\d+(\.|:)', text))

def extract_steps(text):
    # Split the text into sections by numbered items
    split_lines = text.split("\n\n")
    steps = [s for s in split_lines if is_serialized(s)]
    return steps

def store_thoughts_and_steps(full_response):
    store = {}
    store["thoughts"] = full_response.split('</think>')[0].split('<think>')[1:]
    steps = extract_steps(full_response.split('</think>')[-1])
    store["steps"] = steps
    return store

In [6]:
user_input = "2. Do you see any Radius Rejects ?"
full_response = step_outliner(user_input)
store = store_thoughts_and_steps(full_response)

print("Printing Thoughts:")
print(store["thoughts"][0])

print("Printing Steps:")
for step in store["steps"]:
    print(step)
    print("=====================================")

Printing Thoughts:

Okay, so I need to figure out if there are any RADIUS Rejects in the provided pcap file. I remember that RADIUS uses certain codes to indicate the type of message. Specifically, an Access-Accept is code 2, and an Access-Reject is code 3. 

Looking at the data, I see a column named radius.code. I should check the values in this column. If any of the rows have a value of 3, that indicates a RADIUS Reject. 

I'll go through each row one by one. The first row has radius.code 2, which is an Access-Accept. The second row also has radius.code 2. The third row, I'm not sure, but from the data it seems like it's also 2. Wait, let me check again. No, actually, the third row might have a different code. 

Wait, looking back, the third row's radius.code is 2 as well. So none of the first three rows are Rejects. But there are 19 packets in total, so I need to check all of them. Maybe in the later rows, there's a code 3. 

I should also look at the radius.rsp column to see if it'

In [7]:

runner = Task(df, user_query=user_input)
steps = runner.execute(store["steps"])


for (idx,step) in enumerate(steps):
    print("Step:",step)
    runner.router(step)
    print("AFTER EVALUATION:")
    step_answer = runner.steps_eval[idx]["answer"]
    print(step_answer)
    print("=========================================")




user query goal short
Step: 1. **Identify the RADIUS Code Column**: Look for the `radius.code` column in the CSV file. This column indicates the type of RADIUS message.

2. **Understand RADIUS Codes**: 
   - **Code 2**: Access-Accept
   - **Code 3**: Access-Reject

3. **Search for Code 3**: Scroll through the `radius.code` column to find any entries with the value `3`.

4. **Count the Rejects**: Count how many times `3` appears in the `radius.code` column. Each occurrence signifies a RADIUS Reject.
valid_columns ['radius.code']
df_in_markdown |    |   radius.code |
|---:|--------------:|
|  0 |             1 |
|  1 |            11 |
|  2 |             1 |
|  3 |             3 |
|  4 |             1 |
|  5 |            11 |
|  6 |             1 |
|  7 |             2 |
|  8 |             1 |
|  9 |             2 |
| 10 |             1 |
| 11 |             2 |
| 12 |             1 |
| 13 |             3 |
| 14 |             1 |
| 15 |             1 |
| 16 |             1 |
| 17 |        

In [6]:
# Filter the DataFrame for frames with length less than 100 bytes
# Filter the dataframe to include only Radius Request packets (code == 1)
radius_requests = df[df['radius.code'] == 1]

# Check for duplicate udp.payload values
duplicates = radius_requests[radius_requests.duplicated(subset=['udp.payload'], keep=False)]

# Display the results
if not duplicates.empty:
    print("Duplicate Radius Request Packets Found:")
    print(duplicates)
else:
    print("No duplicate Radius Request packets found.")

Duplicate Radius Request Packets Found:
    Unnamed: 0 eap.identity  ip.dsfield.dscp  udp.length udp.checksum  eap.id  \
14          14          NaN                0          83       0xfe66     NaN   
15          15          NaN                0          83       0xfe66     NaN   
16          16          NaN                0          83       0xfe66     NaN   

    udp.time_relative  udp.srcport  frame.time_epoch  frame.time_delta  ...  \
14           0.000000        62956        1440447904         22.189281  ...   
15           5.004208        62956        1440447909          5.004208  ...   
16          10.008839        62956        1440447914          5.004631  ...   

    _ws.expert.group _ws.expert.message eap.md5.value_size eap.md5.value  \
14               NaN                NaN                NaN           NaN   
15               NaN                NaN                NaN           NaN   
16               NaN                NaN                NaN           NaN   

   _ws.expert

In [13]:
import subprocess
def get_ports_radius(pcap_path):
    port_radius = []
    command = f"tshark -r {pcap_path} -T fields -e udp.port"
    res = subprocess.run(command, shell=True, capture_output=True, text=True)
    x = res.stdout.split("\n")[:-1]
    print(x)
    for port_pair in x:
        if port_pair:
            ports = list(map(int, port_pair.split(",")))
            port_radius.append(min(ports))
    return port_radius

def is_radius(pcap_path):
    ports = get_ports_radius(pcap_path)
    if not ports:
        return False
    print(ports)
    command = f"tshark -r {pcap_path} -d udp.port=={ports[0]},radius -Y 'radius'"
    print(command)
    res = subprocess.run(command, shell=True, capture_output=True, text=True)
    print(res.stdout)
    if res.stdout:
        return True
    return False


p1 = "/home/ash/github/packet_analysis/pcap_store/wifi_customer/RoamingIQRadiusfiltered.pcapng"
p2 = "/home/ash/github/packet_analysis/pcap_store/wifi_customer/roamingIQAP1.pcap"
p3 = "/home/ash/github/packet_analysis/pcap_store/wifi_customer/roamingIOairfiltered.pcapng"
print(is_radius(p1))
print(is_radius(p2))
print(is_radius(p3))

['34032,3833', '3833,34032', '39885,3834', '3834,39885']
[3833, 3833, 3834, 3834]
tshark -r /home/ash/github/packet_analysis/pcap_store/wifi_customer/RoamingIQRadiusfiltered.pcapng -d udp.port==3833,radius -Y 'radius'
    1   0.000000 73.233.222.192 → 172.31.65.199 RADIUS 447 Access-Request id=7
    2   0.056584 172.31.65.199 → 73.233.222.192 RADIUS 455 Access-Accept id=7

True
['34032,3833', '3833,34032', '39885,3834', '3834,39885']
[3833, 3833, 3834, 3834]
tshark -r /home/ash/github/packet_analysis/pcap_store/wifi_customer/roamingIQAP1.pcap -d udp.port==3833,radius -Y 'radius'
    1   0.000000 10.0.128.103 → 3.239.90.114 RADIUS 453 Access-Request id=7
    2   0.082559 3.239.90.114 → 10.0.128.103 RADIUS 461 Access-Accept id=7

True
['', '', '', '', '', '', '', '', '', '', '', '', '']
False


In [47]:
context = context
question = "you are tasked with the following job:"+steps[0]+ 'return all possible column names that would be needed to answer the question'

In [48]:
from openai import OpenAI
from pydantic import BaseModel
gpt = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

tools = [{
    "type": "function",
    "function": {
        "name": "get_columns",
        "description": "Get columns and correspondign rows from a dataframe that is required to answer a question",
        "parameters": {
            "type": "object",
            "properties": {
                "columns": {
                    "type": "string",
                    "description": "List of columns required from the dataframe"
                },
                "num_of_rows": {
                    "type": "number",
                    "description": "Number of rows to display for the columns. Select 0 to display all rows"
                },
            },
            "required": [
                "columns",
                "num_of_rows"
            ],
            "additionalProperties": False
        },
        "strict": True
    }
}]

class ListOfColumns(BaseModel):
    columns: list[str]

content = f"""
[context]: {context}
[question]: {question}
"""
print(content)
completion = gpt.beta.chat.completions.parse(
    model="gpt-4o",
    messages=[{"role": "user", "content": content}],
    response_format=ListOfColumns
)

print(completion.choices[0].message)


[context]:  You are provided a pcap file in csv format. These are the columns in the file with first 3 rows:
                |    |   Unnamed: 0 | eap.identity   |   ip.dsfield.dscp |   udp.length | udp.checksum   |   eap.id |   udp.time_relative |   udp.srcport |   frame.time_epoch |   frame.time_delta |   eap.type | ip.dsfield   | radius.authenticator                            | radius.eap_fragment                                               | frame.interface_name   |   ip.dsfield.ecn |   udp.dstport |   null.family |   frame.len |   ip.flags.rb | ip.addr   |   frame.section_number | ip.src_host   |   udp.port | ip.src    |   ip.version | ip.id   |   radius.code |   eap.len |   ip.ttl |   frame.marked |   frame.encap_type | udp.payload                                                                                                                                                                                                                                                         

In [46]:
l = ['Unnamed: 0', 'eap.identity', 'ip.dsfield.dscp', 'udp.length', 'udp.checksum', 'eap.id', 'udp.time_relative', 'udp.srcport', 'frame.time_epoch', 'frame.time_delta', 'eap.type', 'ip.dsfield', 'radius.authenticator', 'radius.eap_fragment', 'frame.interface_name', 'ip.dsfield.ecn', 'udp.dstport', 'null.family', 'frame.len', 'ip.flags.rb', 'ip.addr', 'frame.section_number', 'ip.src_host', 'udp.port', 'ip.src', 'ip.version', 'ip.id', 'radius.code', 'eap.len', 'ip.ttl', 'frame.marked', 'frame.encap_type', 'udp.payload', 'frame.time_utc', 'ip.hdr_len', 'frame.ignored', 'udp.checksum.status', 'radius.avp', 'ip.flags', 'ip.dst', 'eap.code', '_index', 'ip.len', '_type', 'ip.checksum', 'ip.flags.df', 'ip.proto', 'frame.protocols', 'ip.dst_host', 'frame.time', 'frame.interface_id', 'ip.flags.mf', 'udp.stream', 'udp.time_delta', 'frame.time_relative', 'ip.frag_offset', 'radius.length', '_score', 'ip.host', 'ip.checksum.status', 'radius.req', 'frame.cap_len', 'frame.offset_shift', 'radius.id', 'frame.time_delta_displayed', 'radius.avp.length', 'radius.avp.type', 'frame.number', 'radius.time', 'radius.rsp', 'radius.State', 'radius.reqframe', 'eap.mitm_attacks', '_ws.expert.group', '_ws.expert.message', 'eap.md5.value_size', 'eap.md5.value', '_ws.expert.severity', 'radius.Message_Authenticator', 'radius.User_Name', 'radius.Framed_Compression', 'radius.dup', 'radius.req.dup']
print(len(l))

83


In [ ]:
# Upload a file with an "assistants" purpose
file = client.files.create(
file=open(pcap_path, "rb"),
purpose='assistants'
)

# Create an assistant using the file ID
assistant = client.beta.assistants.create(
instructions="You are a network engineer analyzing a pcap file in csv format. Answer the questions based on the provided data.",
model="gpt-4o",
tools=[{"type": "code_interpreter"}],
tool_resources={
  "code_interpreter": {
    "file_ids": [file.id]
  }
}
)

Python REPL can execute arbitrary code. Use with caution.


'2\n'